# Parameter Sweep Demo for Swing Range Expansion Strategy

This notebook demonstrates how to use the parameter sweep utility to optimize strategy parameters and visualize the results.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

try:
    from utils.experiments.sweeper import run_parameter_sweep
    from src.strategies.swing_range_expansion.runner.backtest_runner import SwingRangeExpansionBacktestRunner
    print("✅ All imports successful!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please ensure you're running from the correct directory and all modules are available.")

## Setup Parameter Grid

Define the parameter ranges to sweep over:

In [ ]:
# Define parameter grid for sweep - using correct SwingRangeConfig parameter names
param_grid = {
    'nr_lookback': [5, 7, 10],           # Days to test for narrowest range
    'target_rr': [1.0, 1.5, 2.0],       # × NR range for take-profit
    'stop_rr': [0.5, 0.75, 1.0],        # × NR range for stop-loss
}

print(f"Total combinations: {len(param_grid['nr_lookback']) * len(param_grid['target_rr']) * len(param_grid['stop_rr'])}")
print(f"Parameter grid: {param_grid}")
print(f"\nNote: These parameters match SwingRangeConfig fields:")
print(f"- nr_lookback: {param_grid['nr_lookback']}")
print(f"- target_rr: {param_grid['target_rr']}")
print(f"- stop_rr: {param_grid['stop_rr']}")

## Run Parameter Sweep

Execute the parameter sweep using the utility:

In [ ]:
# Run parameter sweep using the corrected function
print("Starting parameter sweep...")
instrument_ids = ["NIFTY.D.NSE"]  # Example instrument - adjust as needed

# Use the function directly instead of class
try:
    results_df = run_parameter_sweep(
        SwingRangeExpansionBacktestRunner, 
        param_grid, 
        instrument_ids,
        workers=4
    )
    
    print(f"Completed {len(results_df)} parameter combinations")
    print("\nFirst few results:")
    print(results_df.head())
    
    # Check for errors in results
    if 'error' in results_df.columns:
        error_count = results_df['error'].notna().sum()
        if error_count > 0:
            print(f"\n⚠️  {error_count} runs had errors:")
            print(results_df[results_df['error'].notna()]['error'].head())
            
except Exception as e:
    print(f"❌ Error running parameter sweep: {e}")
    print("This might be due to missing data or configuration issues.")
    # Create dummy data for demonstration
    print("\nCreating dummy data for demonstration...")
    np.random.seed(42)
    dummy_data = []
    for nr in param_grid['nr_lookback']:
        for target in param_grid['target_rr']:
            for stop in param_grid['stop_rr']:
                dummy_data.append({
                    'nr_lookback': nr,
                    'target_rr': target,
                    'stop_rr': stop,
                    'return_pct': np.random.normal(0.1, 0.05),
                    'sharpe': np.random.normal(1.2, 0.3),
                    'mdd_pct': np.random.normal(-0.15, 0.05),
                    'win_rate': np.random.uniform(0.4, 0.7),
                    'total_trades': np.random.randint(50, 200)
                })
    results_df = pd.DataFrame(dummy_data)
    print(f"Created {len(results_df)} dummy results for visualization")

## Analyze Results

Convert results to DataFrame and analyze performance:

In [ ]:
# Display basic statistics
print("Parameter Sweep Results Summary:")
print(f"Number of combinations: {len(results_df)}")

# Use the actual column names from the results
return_col = 'return_pct' if 'return_pct' in results_df.columns else 'total_return'
sharpe_col = 'sharpe' if 'sharpe' in results_df.columns else 'sharpe_ratio'

if return_col in results_df.columns:
    print(f"Best Return: {results_df[return_col].max():.2%}")
    print(f"Worst Return: {results_df[return_col].min():.2%}")
    print(f"Average Return: {results_df[return_col].mean():.2%}")

if sharpe_col in results_df.columns:
    print(f"Best Sharpe Ratio: {results_df[sharpe_col].max():.3f}")
    print(f"Average Sharpe Ratio: {results_df[sharpe_col].mean():.3f}")

if 'mdd_pct' in results_df.columns:
    print(f"Lowest MDD: {results_df['mdd_pct'].min():.2%}")
    print(f"Average MDD: {results_df['mdd_pct'].mean():.2%}")

# Show top 5 performers by return
if return_col in results_df.columns:
    print(f"\nTop 5 by {return_col.replace('_', ' ').title()}:")
    cols_to_show = ['nr_lookback', 'target_rr', 'stop_rr', return_col]
    if sharpe_col in results_df.columns:
        cols_to_show.append(sharpe_col)
    if 'mdd_pct' in results_df.columns:
        cols_to_show.append('mdd_pct')
    
    top_performers = results_df.nlargest(5, return_col)[cols_to_show]
    print(top_performers)

## 3D Visualization

Create interactive visualizations to explore parameter relationships:

In [ ]:
# Create 3D scatter plot for parameter relationships
# Use the actual column names from the results
return_col = 'return_pct' if 'return_pct' in results_df.columns else 'total_return'
sharpe_col = 'sharpe' if 'sharpe' in results_df.columns else 'sharpe_ratio'

required_cols = ['nr_lookback', 'target_rr', 'stop_rr', return_col]
if all(col in results_df.columns for col in required_cols):
    fig = go.Figure(data=go.Scatter3d(
        x=results_df['nr_lookback'],
        y=results_df['target_rr'],
        z=results_df['stop_rr'],
        mode='markers',
        marker=dict(
            size=8,
            color=results_df[return_col],
            colorscale='Viridis',
            colorbar=dict(title=return_col.replace('_', ' ').title()),
            showscale=True
        ),
        text=[f"NR Lookback: {nr}<br>Target: {target:.1f}R<br>Stop: {stop:.1f}R<br>Return: {ret:.2%}" + 
              (f"<br>Sharpe: {sr:.3f}" if sharpe_col in results_df.columns else "")
              for nr, target, stop, ret, sr in zip(
                  results_df['nr_lookback'], 
                  results_df['target_rr'], 
                  results_df['stop_rr'], 
                  results_df[return_col], 
                  results_df.get(sharpe_col, [0]*len(results_df))
              )],
        hovertemplate='%{text}<extra></extra>'
    ))
    
    fig.update_layout(
        title=f'Parameter Sweep Results - {return_col.replace("_", " ").title()}',
        scene=dict(
            xaxis_title='NR Lookback (Days)',
            yaxis_title='Target RR',
            zaxis_title='Stop RR'
        ),
        width=800,
        height=600
    )
    
    fig.show()
else:
    print("❌ Required columns not found for 3D visualization")
    print(f"Required: {required_cols}")
    print(f"Available columns: {list(results_df.columns)}")

## Risk-Return Analysis

Analyze the risk-return profile of different parameter combinations:

In [ ]:
# Risk-Return scatter plot
# Use the actual column names from the results
return_col = 'return_pct' if 'return_pct' in results_df.columns else 'total_return'
sharpe_col = 'sharpe' if 'sharpe' in results_df.columns else 'sharpe_ratio'

if all(col in results_df.columns for col in [return_col, 'mdd_pct']):
    color_col = sharpe_col if sharpe_col in results_df.columns else None
    size_col = 'win_rate' if 'win_rate' in results_df.columns else None
    
    fig = px.scatter(
        results_df, 
        x='mdd_pct', 
        y=return_col,
        color=color_col,
        size=size_col,
        hover_data=['nr_lookback', 'target_rr', 'stop_rr'],
        title='Risk-Return Profile',
        labels={
            'mdd_pct': 'Maximum Drawdown (%)',
            return_col: return_col.replace('_', ' ').title() + ' (%)',
            sharpe_col: 'Sharpe Ratio'
        }
    )
    
    fig.update_layout(width=800, height=600)
    fig.show()
    
    # Find the best risk-adjusted returns (highest Sharpe ratio)
    if sharpe_col in results_df.columns:
        best_sharpe = results_df.loc[results_df[sharpe_col].idxmax()]
        print(f"\nBest Risk-Adjusted Performance (Highest Sharpe Ratio):")
        print(f"Parameters: NR Lookback={best_sharpe['nr_lookback']}, Target={best_sharpe['target_rr']:.1f}R, Stop={best_sharpe['stop_rr']:.1f}R")
        print(f"Results: Return={best_sharpe[return_col]:.2%}, Sharpe={best_sharpe[sharpe_col]:.3f}, MDD={best_sharpe['mdd_pct']:.2%}")
else:
    print("❌ Required columns not found for risk-return analysis")
    print(f"Required: {[return_col, 'mdd_pct']}")
    print(f"Available columns: {list(results_df.columns)}")